In [ ]:
import numpy as np
np.random.seed(42)

# ============================================================
# BACKPROPAGATION FROM SCRATCH
# Network: 2 inputs -> 4 hidden (sigmoid) -> 1 output (sigmoid)
# Task: Learn XOR - [0,0]->0, [0,1]->1, [1,0]->1, [1,1]->0
# ============================================================

X = np.array([[0,0], [0,1], [1,0], [1,1]])
y = np.array([[0], [1], [1], [0]])

# Initialize weights (small random values)
W1 = np.random.randn(2, 4) * 0.5  # Input -> Hidden (2x4)
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * 0.5  # Hidden -> Output (4x1)
b2 = np.zeros((1, 1))

def sigmoid(x):
    # Using sigmoid here for pedagogical clarity: its derivative
    # sigmoid(x) * (1 - sigmoid(x)) is simple to implement by hand.
    # Production networks use ReLU, but sigmoid makes backprop math clearer.
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

lr = 1.0  # Learning rate

print("Training backprop from scratch on XOR...")
print("=" * 45)

for epoch in range(5000):
    # ========== FORWARD PASS ==========
    z1 = X @ W1 + b1           # Linear (4,2)@(2,4) = (4,4)
    a1 = sigmoid(z1)           # Activation
    z2 = a1 @ W2 + b2          # Linear (4,4)@(4,1) = (4,1)
    a2 = sigmoid(z2)           # Output prediction

    loss = np.mean((y - a2)**2)  # MSE loss (1/N factor)

    # ========== BACKWARD PASS (Chain Rule!) ==========
    # Note: We use MSE = (1/N)Σ(y-a2)². The 1/N scales the
    # gradient but doesn't change its direction.
    # Step 1: dL/da2 = 2(a2 - y) / n
    dL_da2 = 2 * (a2 - y) / len(y)

    # Step 2: da2/dz2 = sigmoid'(z2) = a2*(1-a2)
    da2_dz2 = a2 * (1 - a2)

    # Step 3: dL/dz2 = dL/da2 * da2/dz2 (chain rule!)
    dL_dz2 = dL_da2 * da2_dz2

    # Step 4: Gradients for W2, b2
    dL_dW2 = a1.T @ dL_dz2
    dL_db2 = np.sum(dL_dz2, axis=0, keepdims=True)

    # Step 5: Propagate to hidden layer
    dL_da1 = dL_dz2 @ W2.T
    da1_dz1 = a1 * (1 - a1)
    dL_dz1 = dL_da1 * da1_dz1

    # Step 6: Gradients for W1, b1
    dL_dW1 = X.T @ dL_dz1
    dL_db1 = np.sum(dL_dz1, axis=0, keepdims=True)

    # ========== UPDATE WEIGHTS ==========
    W2 -= lr * dL_dW2
    b2 -= lr * dL_db2
    W1 -= lr * dL_dW1
    b1 -= lr * dL_db1

    if epoch % 1000 == 0:
        print(f"Epoch {epoch:>4d}: Loss = {loss:.6f}")

print(f"\nFinal predictions: {a2.flatten().round(2)}")
print(f"Targets:           {y.flatten()}")
print("\nNetwork learned XOR using backpropagation!")

In [ ]:
import torch
import torch.nn as nn

# ============================================================
# PYTORCH 2.x AUTOGRAD - Compiled Backpropagation
# XOR problem with modern PyTorch: autograd + torch.compile
# ============================================================

X = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
y = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)

model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),         # Modern: ReLU instead of Sigmoid
    nn.Linear(8, 1),
    nn.Sigmoid()       # Output layer only
)

# Optional: compile model for faster training (PyTorch 2.x)
# compiled_model = torch.compile(model)  # Compiles forward + backward!

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

print("Training with PyTorch autograd...")
print("=" * 45)

for epoch in range(2000):
    # Forward pass (PyTorch records computational graph)
    pred = model(X)
    loss = loss_fn(pred, y)

    # Backward pass (automatic chain rule!)
    optimizer.zero_grad()  # CRITICAL: Clear old gradients
    loss.backward()        # Compute ALL gradients via chain rule
    optimizer.step()       # Update weights using Adam

    if epoch % 400 == 0:
        # Monitor gradient health
        grad_norms = {name: p.grad.norm().item()
                      for name, p in model.named_parameters()
                      if p.grad is not None}
        print(f"Epoch {epoch:>4d}: Loss={loss.item():.6f}, "
              f"Grad norms: {grad_norms}")

# ============================================================
# INFERENCE MODE (faster than no_grad)
# ============================================================
with torch.inference_mode():  # PyTorch 2.x: even faster than no_grad
    preds = model(X)
    print(f"\nPredictions: {preds.flatten().numpy().round(2)}")
    print(f"Targets:     {y.flatten().numpy()}")

# ============================================================
# GRADIENT INSPECTION
# ============================================================
pred = model(X)
loss = loss_fn(pred, y)
optimizer.zero_grad()  # Clear gradients first (essential!)
loss.backward()

print("\nGradient inspection:")
for name, param in model.named_parameters():
    g = param.grad
    print(f"  {name:>10s}: shape={list(param.shape)}, "
          f"norm={g.norm():.4f}, "
          f"has_nan={torch.isnan(g).any().item()}")
print("\nAll gradients computed automatically!")